# Homelessness in Australia: trends, vulnerability and drivers

**Source:** Australian Institute of Health and Welfare, Specialist Homelessness Services monthly data (cat. no. HOU 321), March 2026 release.
**Period:** July 2017 to March 2026, 105 monthly observations.

This notebook goes from the published AIHW workbook to the findings and chart exports. Every figure quoted in the README and in the Tableau dashboard is produced here.

### The question

Australia's specialist homelessness services are funded against a picture of who needs help and why. If that picture has shifted since 2017, the funding and service mix may be calibrated to a problem that no longer looks the way it did. Three sub-questions:

1. **Scale.** Where is demand growing, and is national growth hiding state-level concentration?
2. **Vulnerability.** Is the system serving people before they become homeless, or after?
3. **Drivers.** What is actually bringing people through the door, and has that changed?

### A note on how this collection counts

The SHS collection allows a client to report more than one reason for seeking assistance. This has a direct consequence for analysis: reason categories cannot be summed. AIHW handles this by publishing both individual reasons and an unduplicated total for each reason group. Steps 3 to 5 establish this and it governs every figure that follows.

---
## Phase A: Load and inspect
---

### Step 1. Load the three source sheets

The workbook holds three sheets of interest. Each has two title rows and a blank row above the header, so the header sits on row 4 (`skiprows=3`).

Two cleaning steps are applied on load, and both matter:

- **Trailing note rows.** Each sheet ends with a non-breaking space and a snapshot note in the `Month` column. Coercing `Month` to a datetime and dropping the failures removes them without hard-coding row counts, which would break on the next quarterly release.
- **Mixed-type value columns.** State columns arrive as a mix of numbers and strings. Left uncoerced, any `groupby().sum()` fails with a string concatenation error rather than a type error, which is a confusing way to find out.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

SRC = Path('data/aihw-hou-321-SHS-data-tables_March-2026.xlsx')
OUT = Path('outputs'); OUT.mkdir(exist_ok=True)

STATES = ['NSW', 'Vic', 'Qld', 'WA', 'SA', 'Tas', 'ACT', 'NT']
RC = 'Reason for seeking assistance'

def load_sheet(name):
    """Load one AIHW sheet, drop trailing note rows, coerce value columns to numeric."""
    df = pd.read_excel(SRC, sheet_name=name, skiprows=3)
    df.columns = [str(col).strip() for col in df.columns]
    df['Month'] = pd.to_datetime(df['Month'], errors='coerce')
    df = df[df['Month'].notna()].copy()
    for col in STATES + ['National']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

clients       = load_sheet('Clients')
client_groups = load_sheet('Client Groups')
reasons       = load_sheet('Reasons')

for name, df in [('Clients', clients), ('Client Groups', client_groups), ('Reasons', reasons)]:
    print(f'{name:15s} {df.shape[0]:>6,} rows x {df.shape[1]} cols   '
          f'{df.Month.min():%b %Y} to {df.Month.max():%b %Y}')

Clients          5,040 rows x 12 cols   Jul 2017 to Mar 2026
Client Groups    2,835 rows x 12 cols   Jul 2017 to Mar 2026
Reasons         10,515 rows x 13 cols   Jul 2017 to Mar 2026


### Step 2. Profile the dimensions before filtering anything

Three things need establishing before any subsetting, because each one is a place where a plausible-looking filter produces a wrong answer.

**Sex.** The column holds `Female`, `Male` and `Total`. `Total` is AIHW's own figure, not the sum of the other two, and the three coexist as rows in the same frame. Failing to filter means every total is counted twice over.

**Age group.** Alongside the fifteen age bands sit `All Females`, `All Males` and `All Ages`. Same trap: subtotal rows living beside the detail rows they summarise.

**Group.** The reason groups are not the four in the published dashboard. There are eight: the four visible ones plus `Other`, `Not stated`, `Total` and `COVID`. Which of these appear in a chart is an analytical decision, and one that has to be stated rather than left implied.

In [2]:
print('Sex:      ', list(clients['Sex'].dropna().unique()))
print('Age group:', list(clients['Age group'].dropna().unique()))
print()
print('Reason groups:', list(reasons['Group'].dropna().unique()))
print()
print('Client groups:')
for label in client_groups['Client Group'].dropna().unique():
    print('   -', label)

Sex:       ['Female', 'Male', 'Total']
Age group: ['0–4 years', '5–9 years', '10–14 years', '15–17 years', '18–19 years', '20–24 years', '25–29 years', '30–34 years', '35–39 years', '40–44 years', '45–49 years', '50–54 years', '55–59 years', '60–64 years', '65+ years', 'All Females', 'All Males', 'All Ages']

Reason groups: ['Accommodation', 'Financial', 'Interpersonal', 'Health', 'Other', 'Not stated', 'Total', 'COVID']

Client groups:
   - Number of Indigenous clients
   - Number of clients who have experienced family and domestic violence
   - Number of clients with a current mental health issue
   - Number of clients with problematic drug or alcohol issues
   - Number of clients financially assisted with payments for short term/emergency accommodation
   - Number of nights in short-term/emergency accommodation
   - Number of clients accommodated in short-term/emergency accommodation
   - Number of clients who are homeless
   - Number of clients who are at risk of homelessness


Note the seventh client group: **number of nights** in short-term accommodation. That is a different unit from every other row in the sheet, which count clients. It cannot share an axis with them, and it is excluded from the vulnerability analysis for that reason.

Filtering to the totals now gives the working frames.

In [3]:
# Total clients: Sex = Total AND Age group = All Ages
total_clients = (clients[(clients['Sex'] == 'Total') & (clients['Age group'] == 'All Ages')]
                 .set_index('Month').sort_index())

groups_t  = client_groups[client_groups['Sex'] == 'Total'].copy()
reasons_t = reasons[(reasons['Sex'] == 'Total') & reasons['Group'].notna()].copy()

print(f'Total clients frame: {len(total_clients)} months')
print(f'March 2026 national total: {total_clients["National"].iloc[-1]:,.0f} clients')

Total clients frame: 105 months
March 2026 national total: 100,421 clients


### Step 3. Separate unduplicated group totals from individual reasons

**This is the step the whole analysis turns on.**

Within each group, the Reasons sheet carries one row that is not a reason. It is AIHW's unduplicated count: the number of clients who cited *any* reason in that group, each client counted once. The remaining rows are the individual reasons, and a client reporting both financial difficulties and housing affordability stress appears in two of them.

The consequence is easy to state and easy to miss. **Summing individual reasons does not give the group total.** It gives something larger, and it inflates unevenly, because groups with more sub-reasons accumulate more overlap.

Identifying those total rows needs care. The obvious rule, that the reason name equals the group name, holds for most groups and fails silently for the ones where it does not: Interpersonal's total is labelled `Interpersonal Relationships`, and the whole-collection total is `Total Clients`. A name-match filter leaves those rows sitting in the individual-reason set, where they are large enough to distort any ranking they appear in without looking obviously wrong.

The mapping is therefore declared explicitly and then validated against an invariant that must hold if it is correct: an unduplicated group total can never be smaller than its largest single reason, nor larger than all of them added together.

In [4]:
GROUP_TOTAL_ROW = {
    'Accommodation': 'Accommodation',
    'Financial':     'Financial',
    'Interpersonal': 'Interpersonal Relationships',
    'Health':        'Health',
    'Other':         'Other',
    'Not stated':    'Not stated',
    'Total':         'Total Clients',
}

is_total = reasons_t.apply(lambda r: GROUP_TOTAL_ROW.get(r['Group']) == r[RC], axis=1)
group_totals = reasons_t[is_total].copy()
individual   = reasons_t[~is_total].copy()

print(f'Unduplicated group-total rows: {len(group_totals):,}')
print(f'Individual reason rows:        {len(individual):,}\n')

# Validate: max(reason) <= group total <= sum(reasons), for every group and month
latest = reasons_t['Month'].max()
gt  = group_totals[group_totals.Month == latest].set_index('Group')['National']
agg = individual[individual.Month == latest].groupby('Group')['National'].agg(['max', 'sum', 'count'])
v = agg.join(gt.rename('total')).dropna()
v['valid'] = (v['total'] >= v['max']) & (v['total'] <= v['sum'])
print(v.rename(columns={'max': 'largest_reason', 'sum': 'sum_of_reasons',
                        'count': 'n_reasons'})
      .to_string(float_format=lambda x: f'{x:,.0f}'))
assert v['valid'].all(), 'Group-total mapping failed validation'
print('\nMapping validated.')

Unduplicated group-total rows: 735
Individual reason rows:        2,770

               largest_reason  sum_of_reasons  n_reasons  total  valid
Group                                                                 
Accommodation          33,533          73,298          3 53,271   True
Financial              34,701          84,447          5 48,575   True
Health                 15,749          31,689          4 22,168   True
Interpersonal          36,439          65,884          5 48,752   True
Other                  18,729          36,486          9 30,337   True

Mapping validated.


---
## Phase B: Validate
---

### Step 4. Demonstrate the overlap rather than assuming it

The claim in Step 3 is worth proving on the data instead of taking on trust. For a single month, comparing each group's unduplicated total against the sum of its own sub-reasons gives the size of the overlap directly.

In [5]:
undup = (group_totals[group_totals.Month == latest]
         .set_index('Group')['National'].rename('unduplicated'))
summed = (individual[individual.Month == latest]
          .groupby('Group')['National'].sum().rename('sum_of_reasons'))

check = pd.concat([undup, summed], axis=1).dropna()
check['inflation'] = check.sum_of_reasons / check.unduplicated
check['n_reasons'] = individual.groupby('Group')[RC].nunique()

print(f'{latest:%B %Y}\n')
print(check.sort_values('inflation', ascending=False)
      .to_string(float_format=lambda v: f'{v:,.2f}'))

March 2026

               unduplicated  sum_of_reasons  inflation  n_reasons
Group                                                            
Financial         48,575.00       84,447.00       1.74          5
Health            22,168.00       31,689.00       1.43          4
Accommodation     53,271.00       73,298.00       1.38          3
Interpersonal     48,752.00       65,884.00       1.35          5
Other             30,337.00       36,486.00       1.20          9


The inflation factor tracks the number of sub-reasons, which is the mechanism at work. Any comparison between groups built on summed reasons is measuring category structure as much as client behaviour.

### Step 5. Reconcile reason records against actual client numbers

A second check, at a different level. If reason records were client counts, summing all groups would approximate total clients for the month. It does not, and the size of the gap sets what the axis of any reason chart can honestly be called.

In [6]:
CORE = ['Accommodation', 'Financial', 'Interpersonal', 'Health', 'Other']
records = individual[(individual.Month == latest)
                     & individual.Group.isin(CORE)]['National'].sum()
people  = total_clients.loc[latest, 'National']

print(f'{latest:%B %Y}')
print(f'  Sum of all individual reason records : {records:>10,.0f}')
print(f'  Actual total clients                 : {people:>10,.0f}')
print(f'  Ratio                                : {records/people:>10.2f} reasons per client')

March 2026
  Sum of all individual reason records :    291,804
  Actual total clients                 :    100,421
  Ratio                                :       2.91 reasons per client


Roughly three reason records per client. A chart of reason data is plotting **client-reason records**, not clients, and labelling its axis "clients per month" overstates the population by a factor of about three.

### Step 6. Check coverage, and reconcile National against the states

Two structural checks. First that the period is complete with no missing months, since a gap would silently distort any growth calculation. Second, whether AIHW's `National` column equals the sum of the eight state columns.

It does not, quite. The residual is small but non-zero, and it is not an error: a client supported in more than one jurisdiction within a month is counted once nationally and once in each state. `National` is the deduplicated figure and is used throughout.

In [7]:
months = total_clients.index
expected = pd.date_range(months.min(), months.max(), freq='MS')
print(f'Months present: {len(months)} of {len(expected)} expected')
print(f'Missing: {list(set(expected) - set(months)) or "none"}')

residual = total_clients['National'] - total_clients[STATES].sum(axis=1)
print(f'\nNational minus sum-of-states:')
print(f'  max absolute {residual.abs().max():,.0f}   '
      f'as share of national {residual.abs().max()/total_clients["National"].mean()*100:.3f}%')

Months present: 105 of 105 expected
Missing: none

National minus sum-of-states:
  max absolute 265   as share of national 0.289%


---
## Phase C: Analyse
---

### Step 7. Scale: where is demand growing?

Growth is reported on two bases throughout this notebook, and both are shown because they answer slightly different questions.

**Point to point** compares the first month with the last. It is what a reader assumes "since 2017" means, and it is what the dashboard annotations use. It is also sensitive to whatever happened to be going on in those two particular months.

**Twelve-month averages** compare the first year with the last. Less intuitive, more stable, and the better basis for a headline that has to survive scrutiny.

Where the two diverge materially, that divergence is itself worth knowing.

In [8]:
def growth(series):
    """Return point-to-point and 12-month-average growth for a monthly series."""
    s = series.sort_index()
    first12 = s.iloc[:12].mean()
    last12  = s.iloc[-12:].mean()
    return pd.Series({
        'start': s.iloc[0], 'end': s.iloc[-1],
        'p2p_x': s.iloc[-1] / s.iloc[0],
        'p2p_pct': (s.iloc[-1] / s.iloc[0] - 1) * 100,
        'avg_x': last12 / first12,
        'avg_pct': (last12 / first12 - 1) * 100,
    })

state_growth = (pd.DataFrame({st: growth(total_clients[st]) for st in STATES}).T
                .sort_values('avg_pct', ascending=False))
print(state_growth.to_string(float_format=lambda v: f'{v:,.2f}'))

        start       end  p2p_x  p2p_pct  avg_x  avg_pct
Qld 10,650.00 20,124.00   1.89    88.96   1.75    75.03
WA   5,932.00  7,898.00   1.33    33.14   1.31    31.36
ACT  1,606.00  2,036.00   1.27    26.77   1.23    23.17
NT   2,895.00  4,209.00   1.45    45.39   1.14    13.94
Tas  2,270.00  2,728.00   1.20    20.18   1.12    12.10
NSW 22,019.00 23,755.00   1.08     7.88   1.05     4.87
Vic 30,511.00 33,727.00   1.11    10.54   1.03     2.77
SA   6,308.00  6,170.00   0.98    -2.19   0.98    -2.22


Queensland stands well clear of every other jurisdiction on either basis, growing **1.89 times point to point** and **1.75 times** on twelve-month averages. Western Australia is next at around a third, and South Australia has declined slightly.

The Northern Territory shows why both bases are reported. It looks like the second-fastest grower point to point at +45%, and like a middling one at +14% on averages. The single-month endpoints happen to catch a low July 2017 and a high March 2026 in a small, volatile series. On a base of a few thousand clients, one unusual month moves the headline; the averaged figure is the one to quote.

Two things follow for Queensland. "Nearly doubled" is the accurate description, and anything stronger is not supported by this data. More usefully, national totals conceal the concentration entirely: aggregate demand has grown modestly while a single state absorbed most of the increase. Funding formulas anchored to historical client distribution will understate Queensland's need for as long as they go unrevisited.

### Step 8. Drivers: reason groups on unduplicated totals

Using the unduplicated rows established in Step 3. This is where the published dashboard went wrong, and the corrected picture is different in kind, not just in degree.

In [9]:
grp_series = (group_totals.pivot_table(index='Month', columns='Group',
                                       values='National', aggfunc='sum')
              .sort_index())
core = [g for g in CORE if g in grp_series.columns]

print(pd.DataFrame({g: growth(grp_series[g]) for g in core}).T
      .sort_values('end', ascending=False)
      .to_string(float_format=lambda v: f'{v:,.2f}'))

months_financial_higher = (grp_series['Financial'] > grp_series['Accommodation']).sum()
print(f'\nMonths where Financial exceeds Accommodation: '
      f'{months_financial_higher} of {len(grp_series)}')

                  start       end  p2p_x  p2p_pct  avg_x  avg_pct
Accommodation 45,971.00 53,271.00   1.16    15.88   1.16    15.53
Interpersonal 38,914.00 48,752.00   1.25    25.28   1.16    16.05
Financial     35,150.00 48,575.00   1.38    38.19   1.27    27.49
Other         24,987.00 30,337.00   1.21    21.41   1.12    12.02
Health        18,361.00 22,168.00   1.21    20.73   1.14    14.43

Months where Financial exceeds Accommodation: 0 of 105


**Accommodation is the largest reason group in every one of the 105 months.** Financial reasons have grown faster, 38% against 16% point to point, but have not overtaken it and are not close to doing so. Financial is currently the third-largest group, marginally behind Interpersonal.

The contrast with the summed-reason view is instructive. On summed sub-reasons, Financial appears to pass Accommodation; on AIHW's own unduplicated counts it never does. The mechanism is in Step 4: Financial's five sub-reasons accumulate an inflation factor of 1.74 against Accommodation's 1.38 across three. The apparent crossover measures how many boxes a category has been divided into, not how many people are affected.

### Step 9. Drivers: individual reasons

Individual reasons carry no aggregation, so no double counting reaches them. Growth rates here are directly comparable to one another and are the most defensible figures in the analysis.

In [10]:
core_reasons = individual[individual.Group.isin(CORE)]
ind_series = (core_reasons.pivot_table(index='Month', columns=RC,
                                       values='National', aggfunc='sum')
              .sort_index().dropna(axis=1, how='any'))
print(f'{ind_series.shape[1]} individual reasons ranked\n')

reason_growth = (pd.DataFrame({r: growth(ind_series[r]) for r in ind_series.columns}).T
                 .sort_values('p2p_pct', ascending=False))
print(reason_growth[['start', 'end', 'p2p_pct', 'avg_pct']]
      .to_string(float_format=lambda v: f'{v:,.1f}'))

26 individual reasons ranked

                                                                       start      end  p2p_pct  avg_pct
Housing affordability stress                                        19,527.0 34,701.0     77.7     60.7
Employment difficulties                                              3,095.0  4,500.0     45.4     23.6
Unemployment                                                         7,424.0 10,554.0     42.2     30.7
Medical issues                                                       6,442.0  8,679.0     34.7     25.2
Lack of family and/or community support                             13,949.0 18,729.0     34.3     21.6
Transition from custodial arrangements                               2,054.0  2,751.0     33.9     20.7
Inadequate or inappropriate dwelling conditions                     19,034.0 25,343.0     33.1     26.2
Non-family violence                                                  1,363.0  1,811.0     32.9     13.4
Family and domestic violence      

**Housing affordability stress is the fastest-growing reason in the collection, up 77.7% point to point and 60.8% on twelve-month averages.** It is now the second-largest individual reason after housing crisis, having been fourth in 2017.

This is the finding worth leading on, and it is stronger than the group-level story it replaces for three reasons. It survives the double-counting problem entirely. It names a specific, addressable driver rather than a broad category. And it points at a different policy lever from the one a housing-supply response reaches for: affordability stress is a question of what households can pay, not only of what stock exists.

### Step 10. Vulnerability: prevention or crisis?

The client-group categories mostly overlap, since one person can be recorded as Indigenous, experiencing family and domestic violence, and having a current mental health issue at once. Two of them do not overlap in the same way: clients recorded as homeless and clients recorded as at risk of homelessness are alternative housing statuses at presentation.

They are checked against the client total before being compared, because the comparison is only meaningful if they are close to exhaustive.

In [11]:
cg_series = (groups_t.pivot_table(index='Month', columns='Client Group',
                                  values='National', aggfunc='sum').sort_index())
HOMELESS = 'Number of clients who are homeless'
AT_RISK  = 'Number of clients who are at risk of homelessness'

coverage = (cg_series[HOMELESS] + cg_series[AT_RISK]) / total_clients['National']
print(f'Homeless + at risk as share of all clients: '
      f'{coverage.min()*100:.1f}% to {coverage.max()*100:.1f}%')

gap = (cg_series[HOMELESS] - cg_series[AT_RISK]).rename('gap')
by_year = pd.DataFrame({
    'months_homeless_higher': gap.groupby(gap.index.year).apply(lambda s: (s > 0).sum()),
    'months_observed': gap.groupby(gap.index.year).size(),
    'mean_gap': gap.groupby(gap.index.year).mean(),
})
print()
print(by_year.to_string(float_format=lambda v: f'{v:,.0f}'))
print(f'\nMean gap, last 12 months: {gap.tail(12).mean():,.0f} clients per month')

Homeless + at risk as share of all clients: 91.4% to 96.5%

       months_homeless_higher  months_observed  mean_gap
Month                                                   
2017                        0                6    -3,420
2018                        0               12    -3,306
2019                        0               12    -4,002
2020                        4               12    -1,127
2021                        3               12      -877
2022                        9               12       611
2023                       12               12     1,721
2024                       12               12     4,292
2025                       12               12     5,748
2026                        3                3     4,776

Mean gap, last 12 months: 5,363 clients per month


The two statuses account for 95 to 96% of all clients, so the comparison is close to exhaustive and the ordering is meaningful.

**The relationship inverts in 2022.** Through 2017 to 2019 the system consistently served more people at risk than already homeless. From 2023 onward, clients already homeless outnumber those at risk in every single month, currently by around 5,400 per month.

At-risk clients are the prevention end of the system, where intervention is cheaper and outcomes are generally better. A sustained inversion of this kind describes a service system that has shifted from preventing homelessness to responding to it, which is a capacity signal rather than a demand signal.

---
## Phase D: Export
---

### Step 11. Chart exports

Four charts. The reason-growth ranking carries the headline finding and is the one used at the top of the README.

In [12]:
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 160, 'savefig.bbox': 'tight',
    'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25, 'grid.linewidth': 0.6,
    'axes.titlesize': 13, 'axes.titleweight': 'bold', 'axes.titlepad': 12,
})
ORANGE, BLUE, TEAL, RED, GREY = '#E8912A', '#3B6FA8', '#5FB3A8', '#D6544E', '#9AA0A6'
COVID = pd.Timestamp('2020-03-01')

def covid_line(ax):
    ax.axvline(COVID, color=GREY, ls=':', lw=1.2)
    ax.annotate('COVID-19 (Mar 2020)', xy=(COVID, ax.get_ylim()[0]),
                xytext=(4, 6), textcoords='offset points',
                fontsize=8, color=GREY)

In [13]:
# Chart A - clients by state, Queensland highlighted
fig, ax = plt.subplots(figsize=(11.5, 5.8))
for st in STATES:
    lead = st == 'Qld'
    ax.plot(total_clients.index, total_clients[st], lw=2.6 if lead else 1.0,
            color=TEAL if lead else '#BDC1C6', alpha=1.0 if lead else 0.85,
            zorder=3 if lead else 1)

# Direct end-of-line labels, nudged apart where series sit close together
NUDGE = {'SA': -600, 'WA': 500, 'Tas': 450, 'NT': -450, 'ACT': -100}
for st in STATES:
    y = total_clients[st].iloc[-1]
    lead = st == 'Qld'
    ax.annotate(st, xy=(total_clients.index[-1], y),
                xytext=(7, NUDGE.get(st, 0) / 260), textcoords='offset points',
                va='center', fontsize=9.5,
                color=TEAL if lead else '#80868B',
                fontweight='bold' if lead else 'normal')

covid_line(ax)
q = total_clients['Qld']
label = ('Queensland nearly doubled, from '
         f'{q.iloc[0]:,.0f} to {q.iloc[-1]:,.0f} clients per month.' + chr(10) +
         'Every other state grew by a third or less.')
ax.annotate(label, xy=(pd.Timestamp('2018-02-01'), 16800), fontsize=9.5,
            color=TEAL, fontweight='bold', va='bottom', linespacing=1.6)

ax.set_title('Queensland drives national growth in demand for homelessness support')
ax.set_ylabel('Clients per month'); ax.set_xlabel('')
ax.yaxis.set_major_formatter(lambda v, p: f'{v:,.0f}')
ax.set_xlim(total_clients.index[0], total_clients.index[-1] + pd.Timedelta(days=125))
ax.set_ylim(0, 37500)
fig.savefig(OUT / 'chart_A_state_growth.png'); plt.close(fig)

# Chart B - reason growth ranking (README hero)
top = reason_growth.sort_values('p2p_pct', ascending=False).head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(11, 6))
colours = [ORANGE if r == 'Housing affordability stress' else '#C9CDD2' for r in top.index]
ax.barh(range(len(top)), top['p2p_pct'], color=colours, height=0.72)
ax.set_yticks(range(len(top)))
ax.set_yticklabels([r if len(r) < 48 else r[:45] + '...' for r in top.index], fontsize=9)
for i, v in enumerate(top['p2p_pct']):
    ax.text(v + 1.2, i, f'{v:+.1f}%', va='center', fontsize=8.5,
            color='#3C4043', fontweight='bold' if i == len(top) - 1 else 'normal')
ax.set_title('Housing affordability stress is the fastest-growing reason people seek support')
ax.set_xlabel('Growth in monthly client records, July 2017 to March 2026')
ax.set_xlim(0, top['p2p_pct'].max() * 1.16)
ax.grid(axis='y', visible=False)
fig.savefig(OUT / 'chart_B_reason_growth.png'); plt.close(fig)

In [14]:
# Chart C - reason groups, unduplicated
fig, ax = plt.subplots(figsize=(11, 5.5))
palette = {'Accommodation': BLUE, 'Financial': ORANGE,
           'Interpersonal': TEAL, 'Health': RED, 'Other': GREY}
for g in core:
    ax.plot(grp_series.index, grp_series[g], lw=1.9, color=palette.get(g, GREY), label=g)
covid_line(ax)
ax.set_title('Accommodation remains the largest reason group in every month')
ax.set_ylabel('Clients per month (unduplicated within group)'); ax.set_xlabel('')
ax.yaxis.set_major_formatter(lambda v, p: f'{v:,.0f}')
ax.legend(frameon=False, ncol=5, loc='upper left', fontsize=9)
ax.set_ylim(0, grp_series[core].max().max() * 1.22)
fig.savefig(OUT / 'chart_C_reason_groups.png'); plt.close(fig)

# Chart D - homeless vs at risk
fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(cg_series.index, cg_series[HOMELESS], lw=2.1, color=ORANGE, label='Already homeless')
ax.plot(cg_series.index, cg_series[AT_RISK],  lw=2.1, color=BLUE,   label='At risk of homelessness')
ax.fill_between(cg_series.index, cg_series[AT_RISK], cg_series[HOMELESS],
                where=cg_series[HOMELESS] >= cg_series[AT_RISK],
                color=ORANGE, alpha=0.16, interpolate=True)
covid_line(ax)
ax.annotate(f'Already homeless overtakes at risk from 2022,\ngap now ~{gap.tail(12).mean():,.0f} clients per month',
            xy=(pd.Timestamp('2024-06-01'), cg_series[HOMELESS].max() * 0.80),
            fontsize=9.5, color='#3C4043')
ax.set_title('The system has shifted from prevention to crisis response')
ax.set_ylabel('Clients per month'); ax.set_xlabel('')
ax.yaxis.set_major_formatter(lambda v, p: f'{v:,.0f}')
ax.legend(frameon=False, loc='lower right', fontsize=9)
fig.savefig(OUT / 'chart_D_homeless_vs_atrisk.png'); plt.close(fig)

print('Charts written:')
for p in sorted(OUT.glob('chart_*.png')):
    print(f'  {p.name:36s} {p.stat().st_size/1024:>6.0f} KB')

Charts written:
  chart_A_state_growth.png                165 KB
  chart_B_reason_growth.png                94 KB
  chart_C_reason_groups.png               152 KB
  chart_D_homeless_vs_atrisk.png          158 KB
  chart_E_demand_vs_unmet.png              80 KB
  chart_F_indigenous_ratio.png             77 KB


### Step 12. Tidy exports for Tableau

The three CSVs feed the dashboard. The reasons export carries a `record_type` column marking each row as an unduplicated group total or an individual reason, so the distinction established in Step 3 survives into the visualisation layer rather than having to be remembered.

A dashboard built on this file should filter to one or the other. Mixing them, or summing across `record_type`, reintroduces exactly the double counting this notebook exists to avoid.

In [15]:
LABELS = {
    'Number of Indigenous clients': 'Indigenous',
    'Number of clients who have experienced family and domestic violence': 'FDV',
    'Number of clients with a current mental health issue': 'Mental health',
    'Number of clients with problematic drug or alcohol issues': 'Drug or alcohol issues',
    'Number of clients financially assisted with payments for short term/emergency accommodation':
        'Financial assistance for accommodation',
    'Number of nights in short-term/emergency accommodation': 'Nights in emergency accommodation',
    'Number of clients accommodated in short-term/emergency accommodation': 'Emergency accommodation',
    'Number of clients who are homeless': 'Homeless',
    'Number of clients who are at risk of homelessness': 'At risk of homelessness',
}

# Melt state columns into rows: the long shape Tableau expects
def to_long(df, id_cols, value_name):
    return df.melt(id_vars=id_cols, value_vars=STATES,
                   var_name='State', value_name=value_name)

clients_out = (to_long(total_clients.reset_index()[['Month'] + STATES], ['Month'], 'Clients')
               [['Month', 'State', 'Clients']])

reasons_out = pd.concat([
    group_totals.assign(record_type='group_total'),
    individual.assign(record_type='individual_reason'),
])
reasons_out = (to_long(reasons_out, ['Month', 'Group', RC, 'record_type'], 'Count')
               .rename(columns={RC: 'Reason'})
               [['Month', 'Group', 'Reason', 'State', 'Count', 'record_type']])

groups_out = groups_t.copy()
groups_out['Client_Group'] = groups_out['Client Group'].map(LABELS)
groups_out = (to_long(groups_out, ['Month', 'Client_Group'], 'Count')
              [['Month', 'Client_Group', 'State', 'Count']])

for name, df in [('shs_clients_trend', clients_out),
                 ('shs_reasons', reasons_out),
                 ('shs_client_groups', groups_out)]:
    path = OUT / f'{name}.csv'
    df.to_csv(path, index=False)
    print(f'{path.name:28s} {len(df):>7,} rows')

shs_clients_trend.csv            840 rows
shs_reasons.csv               28,040 rows
shs_client_groups.csv          7,560 rows


---
## Findings

| # | Finding | Figure | Basis |
|---|---------|--------|-------|
| 1 | Housing affordability stress is the fastest-growing reason for seeking support | **+77.7%** (+60.8% smoothed) | Single reason, no aggregation |
| 2 | Queensland has the fastest-growing demand of any state, nearly doubling | **1.89x** (1.75x smoothed) | State client totals |
| 3 | Clients already homeless have outnumbered those at risk in every month since 2023 | **~5,400/month** | Near-exhaustive housing status |
| 4 | Accommodation remains the largest reason group throughout | 105 of 105 months | Unduplicated group totals |

## Limitations

**Service contact, not prevalence.** The SHSC records people who sought and received assistance from a government-funded agency. It does not measure homelessness in the community, and it under-records unmet need, since an agency at capacity may turn someone away without generating a record.

**Cross-state comparison.** States determine their own service models and funding, so differences between jurisdictions reflect what each system delivers and records as much as underlying need. State growth is a change in service contact, not a direct measure of change in homelessness.

**The COVID period.** Jurisdictional pandemic responses differed in ways that affected SHS numbers unevenly. Several New South Wales initiatives fell outside SHSC scope, while Victoria's response centred on short-term accommodation that fell within it. Trend comparisons spanning 2020 to 2022 should be read with that in mind.

**Reasons are not exclusive.** A client may report several reasons. Individual reason figures are directly comparable to each other; group figures use AIHW's unduplicated totals; the two are never mixed.

**Client groups overlap.** Apart from the homeless and at-risk statuses used in Step 10, the client group categories are not mutually exclusive and cannot be summed.

---
## Phase E: A second AIHW source, and why it's needed
---

Every finding so far comes from one workbook: the monthly Specialist Homelessness Services data, cat. no. HOU 321. It counts **service contacts** — a client active in a jurisdiction during a given month. A client returning in three different months is counted in three different months. That's the right measure for service-capacity questions, and it's what Phases A to D use throughout.

AIHW publishes a second, separate workbook: the historical annual tables, cat. no. HOU 343. It counts **unique people per financial year**, deduplicated using a statistical linkage key, so a client with several support periods in a year is counted once. It also includes a measure the monthly file does not: **unassisted requests**, people who approached a service and were not helped at the time.

These two workbooks answer different questions and are not interchangeable. Phase E exists to use the second one correctly, on its own terms, rather than forcing it into the monthly file's shape.

### Step 13. Load the historical workbook

Each sheet in this workbook has its own year range and, in two cases, a documented gap or redefinition partway through. A loader that assumes every sheet matches the first one checked will silently misread the ones that don't — `HIST.YOUNG` has 2024-25 stored out of sequence, after a summary column, because AIHW changed the measure's age range and derivation that year and separated it from the comparable run. The loader below reads each sheet's actual header rather than assuming it.

In [16]:
HSRC = Path('data/AIHW-HOU-343-Specialist-homelessness-services-historical-tables-2011-12-to-2024-25.xlsx')

import warnings
import openpyxl

with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message=r"wmf image format is not supported so the image is being dropped",
        category=UserWarning,
        module=r"openpyxl\.reader\.drawings",
    )
    hwb = openpyxl.load_workbook(HSRC, data_only=True)

def hist_year_cols(sheet):
    """Return {column_index: year_label} for a HIST sheet, reading its own header row
    and excluding summary columns like 'Average annual change'."""
    rows = list(hwb[sheet].iter_rows(values_only=True))
    header = rows[3]
    return {i: v for i, v in enumerate(header) if isinstance(v, str) and '–' in v and 'Average' not in v}, rows

def hist_pull(sheet, match_cols):
    """Pull one row from a HIST sheet as a Series indexed by year label.
    match_cols: dict of {column_index: expected_value} identifying the row, e.g.
    {0: 'National', 1: 'Clients (number)', 2: 'All clients'}."""
    years, rows = hist_year_cols(sheet)
    matches = [r for r in rows[4:] if all(r[i] == v for i, v in match_cols.items())]
    if len(matches) != 1:
        raise ValueError(f'{sheet}: expected 1 matching row for {match_cols}, found {len(matches)}')
    r = matches[0]
    return pd.Series({label: r[i] for i, label in years.items()})

print('Historical workbook sheets:', hwb.sheetnames)

Historical workbook sheets: ['Contents', 'Explanatory notes', 'HIST.CLIENTS', 'HIST.INDIGENOUS', 'HIST.INDIGENOUS_REG', 'HIST.REG', 'HIST.YOUNG', 'HIST.CPO', 'HIST.LCARE', 'HIST.EXIT', 'HIST.OLDER', 'HIST.ADF', 'HIST.FDV', 'HIST.DIS', 'HIST.MH', 'HIST.SUB', 'HIST.UNASSISTED']


### Step 14. National unique clients vs monthly service contacts

The two collections measure different things. Placed side by side, the gap between them is itself the finding: whether recorded volume is growing because more people need help, or because the same population is being served more intensively without capacity to match.

In [17]:
hist_clients = hist_pull('HIST.CLIENTS', {0: 'National', 1: 'Clients (number)', 2: 'All clients'})
hist_clients_state = {
    st: hist_pull('HIST.CLIENTS', {0: st, 1: 'Clients (number)', 2: 'All clients'})
    for st in ['New South Wales', 'Victoria', 'Queensland', 'Western Australia',
               'South Australia', 'Tasmania', 'Australian Capital Territory', 'Northern Territory']
}

print('National unique clients per financial year:')
print(hist_clients.to_string())

start, end = '2017–18', '2024–25'
nat_growth = (hist_clients[end] / hist_clients[start] - 1) * 100
print(f"\n{start} -> {end}: {hist_clients[start]:,.0f} -> {hist_clients[end]:,.0f}  ({nat_growth:+.1f}%)")

# Validation: this must match the audit finding before anything downstream uses it.
assert abs(nat_growth - 0.1) < 0.2, f'National unique-client growth drifted from audited +0.1%: got {nat_growth:.2f}%'
print('Validated against audit figure.')

National unique clients per financial year:
2011–12    236429
2012–13    244176
2013–14    254001
2014–15    255657
2015–16    279196
2016–17    288273
2017–18    288795
2018–19    290317
2019–20    290462
2020–21    278275
2021–22    272694
2022–23    273648
2023–24    280078
2024–25    288970

2017–18 -> 2024–25: 288,795 -> 288,970  (+0.1%)
Validated against audit figure.


### Step 15. Unassisted requests: the measure the monthly file doesn't have

An unassisted request is an instance where someone approached a service and was not helped at the time. AIHW does not deduplicate this measure the way it deduplicates clients — the statistical linkage key needed to identify repeat individuals was missing for 45 to 48% of unassisted requests across these years, so AIHW itself reports the figure as request volume, not a person count. That limitation is carried through explicitly below rather than treated as a precise headcount.

In [18]:
hist_unassisted = hist_pull('HIST.UNASSISTED', {0: 'National', 1: 'Number of unassisted requests'})
hist_approach_rate = hist_pull('HIST.UNASSISTED',
    {0: 'National', 1: 'Average number of times a person approached an agency'})

print('National unassisted requests, by year:')
print(hist_unassisted.to_string())

u_growth = (hist_unassisted[end] / hist_unassisted[start] - 1) * 100
print(f"\nRequest growth {start} -> {end}: {u_growth:+.1f}%")

# The approach rate rose too, so raw request growth overstates growth in people turned away.
# Adjust by the change in average approaches per person to get a defensible range, not a
# single spuriously precise number.
approach_growth = (hist_approach_rate[end] / hist_approach_rate[start] - 1) * 100
people_growth_adj = ((hist_unassisted[end] / hist_approach_rate[end]) /
                      (hist_unassisted[start] / hist_approach_rate[start]) - 1) * 100
print(f'Average approaches per person: {hist_approach_rate[start]:.2f} -> {hist_approach_rate[end]:.2f}  ({approach_growth:+.1f}%)')
print(f'Approach-adjusted growth in people turned away (approximate): {people_growth_adj:+.1f}%')
print(f'\nReport as: unassisted requests grew {u_growth:.0f}%; adjusting for the rise in repeat '
      f'approaches, growth in distinct people turned away is closer to {people_growth_adj:.0f}%.')

assisted_growth = nat_growth
unmet_share_start = hist_unassisted[start] / (hist_unassisted[start] + hist_clients[start]) * 100
unmet_share_end = hist_unassisted[end] / (hist_unassisted[end] + hist_clients[end]) * 100
print(f'\nUnmet share of all requests: {unmet_share_start:.1f}% -> {unmet_share_end:.1f}%')

assert abs(unmet_share_end - 30.8) < 0.3, f'Unmet share drifted from audited 30.8%: got {unmet_share_end:.2f}%'
print('Validated against audit figure.')

National unassisted requests, by year:
2017–18     86103
2018–19     92292
2019–20     95252
2020–21    114026
2021–22    105085
2022–23    107520
2023–24    109807
2024–25    128915

Request growth 2017–18 -> 2024–25: +49.7%
Average approaches per person: 1.50 -> 1.60  (+6.7%)
Approach-adjusted growth in people turned away (approximate): +40.4%

Report as: unassisted requests grew 50%; adjusting for the rise in repeat approaches, growth in distinct people turned away is closer to 40%.

Unmet share of all requests: 23.0% -> 30.8%
Validated against audit figure.


### Step 16. Unmet demand by state — comparable at the national level only

AIHW's jurisdiction-specific technical notes name a distinct data or service-model issue for every single state in this collection: Victoria's family-violence intake was deliberately shifted to a non-SHS service across this period; Queensland's QHIP platform and 2022-23 funding increase both affect its unassisted count; South Australia's own service model and client management system are stated by AIHW to under-record unmet need, with South Australia itself saying this does not reflect lower demand; Tasmania's data is suppressed in places and its service model changed in 2024-25.

The state-by-state unmet rate is still computed and exported, because it's the clearest single indicator of where a service system is under the most pressure — but every state carries a named caveat, and it is reported as such rather than as a clean ranking.

In [19]:
STATE_CAVEATS = {
    'New South Wales': "2014-15 service transition affects early-period comparability.",
    'Victoria': "Client numbers affected by a deliberate, ongoing shift of family violence "
                "intake to a non-SHS service; falling client numbers reflect this, not "
                "necessarily falling need.",
    'Queensland': "Unassisted counts affected by the QHIP referral platform; assisted client "
                  "growth partly driven by a 2022-23 and 2023-24 funding increase.",
    'Western Australia': "No jurisdiction-specific caveat identified in AIHW's technical notes "
                          "for this measure.",
    'South Australia': "AIHW states SA's unmet-need and unassisted figures are understated by "
                        "service model and system design; SA itself does not read its low "
                        "unassisted numbers as low demand.",
    'Tasmania': "Data suppressed in some tables; 2024-25 service model change breaks "
                "time series continuity.",
    'Australian Capital Territory': "2016-17 central-intake model affects unassisted-request "
                                     "comparability specifically.",
    'Northern Territory': "A major new agency began reporting from January 2019; full impact "
                           "only evident from 2019-20 onward.",
}

hist_unassisted_state = {
    st: hist_pull('HIST.UNASSISTED', {0: st, 1: 'Number of unassisted requests'})
    for st in hist_clients_state
}

rows = []
for st in hist_clients_state:
    u, c = hist_unassisted_state[st], hist_clients_state[st]
    rate0 = u[start] / (u[start] + c[start]) * 100
    rate1 = u[end] / (u[end] + c[end]) * 100
    rows.append({
        'State': st,
        f'unmet_rate_{start}': rate0, f'unmet_rate_{end}': rate1,
        'change_pp': rate1 - rate0,
        'caveat': STATE_CAVEATS[st],
    })
state_unmet = pd.DataFrame(rows).set_index('State').sort_values(f'unmet_rate_{end}', ascending=False)
print(state_unmet[[f'unmet_rate_{start}', f'unmet_rate_{end}', 'change_pp']]
      .to_string(float_format=lambda v: f'{v:.1f}'))
print()
for st, row in state_unmet.iterrows():
    print(f'{st}: {row.caveat}')

                              unmet_rate_2017–18  unmet_rate_2024–25  change_pp
State                                                                          
Tasmania                                    61.4                64.4        3.0
Western Australia                           46.6                57.6       11.0
Northern Territory                          36.8                50.2       13.3
New South Wales                             13.8                24.1       10.3
Victoria                                    21.8                24.0        2.1
Queensland                                  10.2                22.4       12.2
Australian Capital Territory                10.5                 7.5       -3.0
South Australia                              1.7                 5.0        3.4

Tasmania: Data suppressed in some tables; 2024-25 service model change breaks time series continuity.
Western Australia: No jurisdiction-specific caveat identified in AIHW's technical notes for this 

### Step 17. Older clients and Indigenous clients by remoteness

Two findings independent of the state-comparability problem above, since both are reported at the national level.

In [20]:
hist_older = hist_pull('HIST.OLDER', {0: 'National', 1: 'Clients (number)', 2: 'All clients'})
older_growth = (hist_older[end] / hist_older[start] - 1) * 100
older_share0 = hist_older[start] / hist_clients[start] * 100
older_share1 = hist_older[end] / hist_clients[end] * 100
print(f'Older clients, National: {hist_older[start]:,.0f} ({older_share0:.1f}% of clients) -> '
      f'{hist_older[end]:,.0f} ({older_share1:.1f}%)   growth {older_growth:+.1f}%')

assert abs(older_growth - 31.7) < 0.3, f'Older-client growth drifted from audited +31.7%: got {older_growth:.2f}%'
print('Validated against audit figure.')

Older clients, National: 24,094 (8.3% of clients) -> 31,729 (11.0%)   growth +31.7%
Validated against audit figure.


In [21]:
def hist_reg_rate(status, remoteness):
    years, rows = hist_year_cols('HIST.INDIGENOUS_REG')
    r = [x for x in rows[4:] if x[0] == 'Clients (per 10,000 ERP)' and x[1] == status and x[2] == remoteness]
    return pd.Series({label: r[0][i] for i, label in years.items()})

indig_end = '2024–25(b)'   # AIHW marks this year as provisional for the regional-population base
remoteness_growth = {}
for rc in ['Major cities', 'Inner/Outer regional', 'Remote/Very remote']:
    s = hist_reg_rate('Indigenous clients', rc)
    g = (s[indig_end] / s[start] - 1) * 100
    remoteness_growth[rc] = g
    print(f'{rc:22s} {s[start]:>7.1f} -> {s[indig_end]:>7.1f} per 10,000   ({g:+.1f}%)')

assert remoteness_growth['Remote/Very remote'] > remoteness_growth['Major cities'], \
    'Expected remote areas to show faster growth than major cities — audit finding not reproduced'
print('\nRemote and very remote areas grew fastest, consistent with the audit finding.')

Major cities             596.2 ->   735.8 per 10,000   (+23.4%)
Inner/Outer regional     722.0 ->   776.3 per 10,000   (+7.5%)
Remote/Very remote       732.5 ->  1021.7 per 10,000   (+39.5%)

Remote and very remote areas grew fastest, consistent with the audit finding.


### Step 18. A caveat that only appears on close reading: consent and Indigenous status

Prior to August 2023, clients could decline consent for seven data items to be sent to AIHW, including Indigenous status. That consent requirement was removed in October 2022. If declining previously suppressed some Indigenous-status recording, removing the requirement would lift the recorded count independent of any change in who is presenting. This needs checking against the data before the Indigenous growth finding is reported without qualification.

In [22]:
hist_notstated = hist_pull('HIST.INDIGENOUS', {0: 'National', 1: 'Clients (number)', 2: 'Not stated', 3: 'All clients'})
print("Clients with Indigenous status recorded as 'Not stated', National:")
print(hist_notstated.to_string())

drop_2324 = (hist_notstated['2023–24'] / hist_notstated['2022–23'] - 1) * 100
print(f"\n'Not stated' fell {drop_2324:.1f}% from 2022-23 to 2023-24, the year after consent was "
      f"removed — consistent with a real, one-off reporting effect landing in that year.")

pre_avg = hist_clients_pct = (hist_pull('HIST.INDIGENOUS', {0:'National',1:'Clients (number)',2:'Indigenous clients',3:'All clients'})
                               .pct_change() * 100)
pre = pre_avg.loc['2018–19':'2021–22'].mean()
post = pre_avg.loc['2023–24':'2024–25'].mean()
print(f'\nAverage annual Indigenous client growth, 2018-19 to 2021-22 (pre-change): {pre:+.1f}%')
print(f'Average annual Indigenous client growth, 2023-24 to 2024-25 (post-change): {post:+.1f}%')
print('\nGrowth predates the policy change by several years and continues after the one-off '
      "reporting effect, so the underlying trend is real — but recent-year growth is likely "
      'somewhat overstated by the consent change and should be reported with this caveat attached.')

Clients with Indigenous status recorded as 'Not stated', National:
2011–12    35560
2012–13    36830
2013–14    37100
2014–15    28401
2015–16    27008
2016–17    26941
2017–18    29539
2018–19    25471
2019–20    22141
2020–21    17825
2021–22    16208
2022–23    14732
2023–24     7950
2024–25     7257

'Not stated' fell -46.0% from 2022-23 to 2023-24, the year after consent was removed — consistent with a real, one-off reporting effect landing in that year.

Average annual Indigenous client growth, 2018-19 to 2021-22 (pre-change): +2.9%
Average annual Indigenous client growth, 2023-24 to 2024-25 (post-change): +5.3%

Growth predates the policy change by several years and continues after the one-off reporting effect, so the underlying trend is real — but recent-year growth is likely somewhat overstated by the consent change and should be reported with this caveat attached.


### Step 19. Export for Tableau

Three exports. Each is annotated with its counting basis in the filename or a column, so it cannot be silently combined with the monthly-collection exports from Phase D.

In [23]:
# National demand: monthly service contacts vs annual unique clients, same measure two ways
national_demand = pd.DataFrame({
    'Year': list(hist_clients.index),
    'Unique_clients_annual': hist_clients.values,
    'Unassisted_requests_annual': hist_unassisted.reindex(hist_clients.index).values,
})
national_demand.to_csv(OUT / 'hist_national_demand.csv', index=False)

# Unmet-need rate by state, with the caveat carried as a column rather than left in prose only
state_unmet_out = state_unmet.reset_index()
state_unmet_out.to_csv(OUT / 'hist_state_unmet_rate.csv', index=False)

# Older clients and Indigenous-by-remoteness, national
demographics = pd.DataFrame({
    'Year': list(hist_older.index),
    'Older_clients': hist_older.values,
    'Total_clients': hist_clients.reindex(hist_older.index).values,
})
demographics.to_csv(OUT / 'hist_older_clients.csv', index=False)

indig_reg_out = pd.DataFrame({
    rc: hist_reg_rate('Indigenous clients', rc) for rc in
    ['Major cities', 'Inner/Outer regional', 'Remote/Very remote']
}).reset_index().rename(columns={'index': 'Year'})
indig_reg_out.to_csv(OUT / 'hist_indigenous_by_remoteness.csv', index=False)

for f in ['hist_national_demand', 'hist_state_unmet_rate', 'hist_older_clients', 'hist_indigenous_by_remoteness']:
    p = OUT / f'{f}.csv'
    print(f'{p.name:36s} {len(pd.read_csv(p)):>4} rows')

hist_national_demand.csv               14 rows
hist_state_unmet_rate.csv               8 rows
hist_older_clients.csv                 14 rows
hist_indigenous_by_remoteness.csv       8 rows


---
## Phase E findings

| # | Finding | Figure | Caveat |
|---|---------|--------|--------|
| 5 | Unique clients assisted nationally are essentially flat since 2017-18 | +0.1% | None material |
| 6 | Unassisted requests grew far faster than assisted clients over the same period | +49.7% requests (~40-45% adjusted for repeat approaches) | Linkage key missing for 45-48% of requests; reported as request volume |
| 7 | The share of all requests going unmet has risen | 23.0% → 30.8% | National measure; state-level rates carry jurisdiction-specific caveats (Step 16) |
| 8 | Older clients are a growing share of the system | +31.7%, 8.3% → 11.0% of clients | None material |
| 9 | Indigenous demand is growing fastest in remote Australia | Major cities +23.4%, remote/very remote +39.5% | Recent-year growth partly reflects the October 2022 consent policy change (Step 18) |